### Ready to Go? Let's Start Your Chemical Discovery with CSU-IR !

💡 **Pro Tip:** Want to speed things up? You can use a powerful T4 GPU for free!

Just click on **Runtime** at the top of the page, select **Change runtime type**, and choose **T4 GPU** from the *Hardware accelerator* dropdown.

If a GPU is not available, you can proceed using a CPU runtime. However, please keep the following in mind:
*   All tasks will take longer to complete.

<hr style="border: none; border-top: 2px dashed #ccc;">

#### 🚀 Final Pre-Flight Check (Crucial!)

1.  **Check Your Connection Status** at the top-right of the page.
    - Colab often connects automatically to your last-used device. If you don't see `Connecting...`, please click the **`Connect`** button manually.
2.  **Wait for the Green Checkmark** (✓) to appear next to your `RAM` and `Disk` status. This confirms you are **"Connected"**.
3.  When Colab asks for permission to access your files, go ahead and click **"Connect to Google Drive"** to authorize it.
4.  Once connected, Run Cells **"Step-by-Step"**.

<hr style="border: none; border-top: 2px dashed #ccc;">

#### ✨ After running, get ready to see:
*   **1.NPS Retrieval Result**.
*   **2.Other test results within simulated data**.

---

### **Step 1: Project Setup & Data Download** ⚙️

<div style="background-color: #e6f7ff; border-left: 6px solid #1890ff; padding: 15px; margin: 15px 0; border-radius: 5px;">
    <h4 style="color: #0050b3; margin-top: 0; font-size: 1.1em;">
    </h4>
    <p>
        This first code cell is the starting point for your entire discovery journey. It will automatically handle all the essential preparations for you.
    </p>
    <p>
        <strong>What it does:</strong>
    </p>
    <ul style="margin-top: 0; padding-left: 20px;">
        <li>Connects to your Google Drive</strong> to save and access data.</li>
        <li>Clones or updates our GitHub repository</strong> to ensure you're using the latest code.</li>
        <li>Installs all required Python libraries</strong> to power the analysis.</li>
        <li>Downloads all model weights and datasets</strong> from Hugging Face.</li>
    </ul>
    <p style="font-weight: bold;">
        To begin, simply run the code cell directly below this one.
    </p>
</div>

In [ ]:
#@title 🧠 1: Setting Up Your Workspace
# ==============================================================================
#  CSU-IR Project Setup Script for Colab
# ==============================================================================
import os
import shutil
from google.colab import drive
from google.colab import userdata
from huggingface_hub import hf_hub_download # Import at the top

# ==================================================================
# --- 1. Mount Google Drive ---
# ==================================================================
print("📁 Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True) # Use force_remount for robustness
print("✅ Google Drive mounted.")


#!rm -rf "/content/drive/MyDrive/colab_repos/CSU-IR"



# ==================================================================
# --- 3. Clone or Update Repository (Improved) ---
# ==================================================================

print("\n--- Defining Project Paths ---")
# Project Root
GDRIVE_REPO_PATH = "/content/drive/MyDrive/colab_repos/CSU-IR"

print("\n--- Cloning or Updating Repository ---")
try:
    GIT_TOKEN = userdata.get('GITHUB_PAT')
except userdata.SecretNotFoundError:
    raise Exception("🛑 Error: Please set 'GITHUB_PAT' in Colab Secrets.")

GIT_TOKEN = GIT_TOKEN.strip()
REPO_URL = f"https://{GIT_TOKEN}@github.com/Hsqcsu/CSU-IR.git"

if not os.path.exists(GDRIVE_REPO_PATH):
    print(f"⏳ Cloning to: {GDRIVE_REPO_PATH}")
    !git clone {REPO_URL} {GDRIVE_REPO_PATH}
else:
    print(f"✅ Repository exists. Updating...")
    !cd {GDRIVE_REPO_PATH} && git remote set-url origin {REPO_URL}
    pull_result = !cd {GDRIVE_REPO_PATH} && git pull
    if "fatal" in "".join(pull_result):
        print("⚠️ Pull failed. Trying to reset and pull...")
        !cd {GDRIVE_REPO_PATH} && git fetch --all && git reset --hard origin/main


# ==================================================================
# --- 2. Define All Project Paths ---
# ==================================================================

# Requirements file
REQUIREMENTS_FILE = os.path.join(GDRIVE_REPO_PATH, 'requirements/requirements_colab.txt')

# --- Data and Model Destination Folders on Google Drive ---
# These are the folders where downloaded files will be saved.
#DEST_Multimodal_Spectroscopic_Library_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'data', 'test_data', 'Multimodal_Spectroscopic_Library')
DEST_QM9S_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'data', 'test_data', 'QM9S')
DEST_QMe14S_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'data', 'test_data', 'QMe14S')
DEST_PS_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'data', 'test_data', 'NPS')
DEST_LIB_PS_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'data', 'processed_library', 'PS')

DEST_Stage_I_MODEL_WEIGHTS_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'check_points','Multi-stage_training_Stage_I_MD')
DEST_Stage_II_MODEL_WEIGHTS_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'check_points','Multi-stage_training_Stage_II_DFT')
DEST_Stage_III_MODEL_WEIGHTS_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR', 'check_points','Multi-stage_training_Stage_III_EXP')
print("✅ Paths defined.")


# ==================================================================
# --- 4. Install Dependencies ---
# ==================================================================
print("\n--- Installing Dependencies ---")
if os.path.exists(REQUIREMENTS_FILE):
    print("⏳ Installing dependencies from requirements_colab.txt...")
    !pip install -q -r {REQUIREMENTS_FILE}
    print("✅ Dependencies installed.")
else:
    print(f"⚠️ Warning: Could not find requirements file at {REQUIREMENTS_FILE}.")


# ==================================================================
# --- 5. Define Download Helper and File Lists ---
# ==================================================================
print("\n--- Preparing for Download ---")
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    raise Exception("🛑 Error: Please set your Hugging Face Token in Colab Secrets (secret name: HF_TOKEN)")

# --- Reusable Download Helper Function ---
def download_files_from_hf(repo_id, files_dict, destination_dir, token):
    """
    Downloads a dictionary of files from a Hugging Face repo to a destination directory.
    """
    print(f"\n--- Checking files for: {os.path.basename(destination_dir)} ---")
    os.makedirs(destination_dir, exist_ok=True)

    for hf_path, local_name in files_dict.items():
        target_path = os.path.join(destination_dir, local_name)
        if not os.path.exists(target_path):
            print(f"  -> Downloading '{local_name}'...")
            try:
                downloaded_path = hf_hub_download(repo_id=repo_id, filename=hf_path, token=token)
                shutil.copy(downloaded_path, target_path)
                print(f"  ✅ Downloaded successfully.")
            except Exception as e:
                print(f"  ❌ FAILED to download '{local_name}'. Error: {e}")
        else:
            print(f"  👍 '{local_name}' already exists.")

# --- Define All Files to be Downloaded ---
HF_REPO_ID = "Skylight666/CSU-IR"

Multimodal_Spectroscopic_Library_DATA_TO_DOWNLOAD = {
    "Multi-stage training data/Stage I - MD/Multimodal_Spectroscopic_Library_test_ir.pt": "Multimodal_Spectroscopic_Library_test_ir.pt",
    "Multi-stage training data/Stage I - MD/Multimodal_Spectroscopic_Library_test_smiles.txt": "Multimodal_Spectroscopic_Library_test_smiles.txt",
}
QM9S_DATA_TO_DOWNLOAD = {
    "Multi-stage training data/Stage II- DFT/QM9S_DFT_test_ir.pt": "QM9S_DFT_test_ir.pt",
    "Multi-stage training data/Stage II- DFT/QM9S_DFT_test_smiles.txt": "QM9S_DFT_test_smiles.txt",
}
QMe14S_DATA_TO_DOWNLOAD = {
    "QM9-like molecules in QMe14S(excluding QM9S)/QM9-like molecules in QMe14S(excluding QM9S).pt": "QM9-like molecules in QMe14S(excluding QM9S).pt",
    "QM9-like molecules in QMe14S(excluding QM9S)/QM9-like molecules in QMe14S(excluding QM9S).txt": "QM9-like molecules in QMe14S(excluding QM9S).txt",
}


NPS_DATA_TO_DOWNLOAD = {
    "NPS_data/NPS_ir.pt": "NPS_ir.pt",
    "NPS_data/NPS_smiles.txt": "NPS_smiles.txt",
    "NPS_data/filtered_final_NPS_ir.pt": "filtered_final_NPS_ir.pt",
    "NPS_data/filtered_final_NPS_smiles.txt": "filtered_final_NPS_smiles.txt",
}
PROCESSED_LIBRARY_PS_TO_DOWNLOAD = {
    "Processed_Retrieval_Library/PS_Retrieval/library_Derivative_PS_smiles_features_fp16.gz": "library_Derivative_PS_smiles_features_fp16.gz",
    "Processed_Retrieval_Library/PS_Retrieval/library_Existing_PS_smiles_features_fp16.gz": "library_Existing_PS_smiles_features_fp16.gz",
    "Processed_Retrieval_Library/PS_Retrieval/smiles_Derivative_PS.txt": "smiles_Derivative_PS.txt",
    "Processed_Retrieval_Library/PS_Retrieval/smiles_Existing_PS.txt": "smiles_Existing_PS.txt",
}


MODEL_Stage_I_WEIGHTS_TO_DOWNLOAD = {
    "Model_weights/CSU-IR/Stage-I/best_ir_model_epoch_79_ratio_0.9810.pth": "best_ir_model_epoch_79_ratio_0.9810.pth",
    "Model_weights/CSU-IR/Stage-I/best_smiles_model_epoch_79_ratio_0.9810.pth": "best_smiles_model_epoch_79_ratio_0.9810.pth",
}
MODEL_Stage_II_WEIGHTS_TO_DOWNLOAD = {
    "Model_weights/CSU-IR/Stage-II/best_ir_model_epoch_71_ratio_0.9922.pth": "best_ir_model_epoch_71_ratio_0.9922.pth",
    "Model_weights/CSU-IR/Stage-II/best_smiles_model_epoch_71_ratio_0.9922.pth": "best_smiles_model_epoch_71_ratio_0.9922.pth",
}
MODEL_Stage_III_WEIGHTS_TO_DOWNLOAD = {
    "Model_weights/CSU-IR/Stage-III/best_ir_model_0.9230379746835443.pth": "best_ir_model_0.9230379746835443.pth",
    "Model_weights/CSU-IR/Stage-III/best_smiles_model_0.9230379746835443.pth": "best_smiles_model_0.9230379746835443.pth",
}
print("✅ Download helper and file lists are ready.")

# ==================================================================
# --- 6. Execute All Downloads ---
# ==================================================================
print("\n--- Executing All Downloads ---")
#download_files_from_hf(HF_REPO_ID, Multimodal_Spectroscopic_Library_DATA_TO_DOWNLOAD, DEST_Multimodal_Spectroscopic_Library_PATH, HF_TOKEN)
download_files_from_hf(HF_REPO_ID, QM9S_DATA_TO_DOWNLOAD, DEST_QM9S_PATH, HF_TOKEN)
download_files_from_hf(HF_REPO_ID, QMe14S_DATA_TO_DOWNLOAD, DEST_QMe14S_PATH, HF_TOKEN)
download_files_from_hf(HF_REPO_ID, NPS_DATA_TO_DOWNLOAD, DEST_PS_PATH, HF_TOKEN)
download_files_from_hf(HF_REPO_ID, PROCESSED_LIBRARY_PS_TO_DOWNLOAD, DEST_LIB_PS_PATH, HF_TOKEN)


download_files_from_hf(HF_REPO_ID, MODEL_Stage_III_WEIGHTS_TO_DOWNLOAD, DEST_Stage_III_MODEL_WEIGHTS_PATH, HF_TOKEN)
download_files_from_hf(HF_REPO_ID, MODEL_Stage_II_WEIGHTS_TO_DOWNLOAD, DEST_Stage_II_MODEL_WEIGHTS_PATH, HF_TOKEN)
download_files_from_hf(HF_REPO_ID, MODEL_Stage_I_WEIGHTS_TO_DOWNLOAD, DEST_Stage_I_MODEL_WEIGHTS_PATH, HF_TOKEN)



# ==================================================================
# --- Finalization ---
# ==================================================================
print("\n\n🎉🎉🎉 Full project setup is complete! 🎉🎉🎉")
print('--------------------------------------------------------------------------------------')
print("\nInstallation complete. The runtime will now restart automatically.")
print("Please wait for the session to reconnect and then run the next cell.")

# This command will automatically restart the Colab runtime.
#import os
#os.kill(os.getpid(), 9)
#print("After re-connected, you can run the analysis and training notebooks.")

---
### **Step 2: Model Initialization & Data Loading**
<div style="background-color: #e6f7ff; border-left: 6px solid #1890ff; padding: 15px; margin: 15px 0; border-radius: 5px;">
    <h4 style="color: #0050b3; margin-top: 0;">
        💡 A Note on Initialization
    </h4>
    <p>
        After you run this cell, you might see some error messages appear in the output. <strong>Please don't worry, this is expected.</strong>
    </p>
    <p>
        We have already handled this initialization process within the code. You don't need to do anything.
    </p>
    <p>
        <strong>Simply wait for this cell to finish running, and then proceed to the next cell.</strong>
    </p>
</div>


In [ ]:
#@title 🧠 2. Global Initialization (Load All Models & Data)
print("⏳ Initializing and warming up the RDKit environment...")
try:
    from rdkit import Chem
    # If the first import succeeds, great.
except ImportError as e:
    # The first import may fail in a freshly restarted Colab runtime.
    # This is a known issue. We capture the error and retry,
    # as the failed attempt often prepares the environment for the next one.
    print(f"Caught an expected initialization error: {e}")
    print("Retrying import...")
    from rdkit import Chem

print("✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅✅")
print("The code is running normally")
print("💡💡💡 This is a normal phenomenon, please don't be surprised, wait for the program to finish running and run the next cell.!")
print("-------------------------------------------------------------------------")
print("✅ RDKit environment is ready!")

import sys
import os
import gzip
import torch
from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from rdkit.Chem import Draw, MolFromSmiles
from google.colab import files
import pandas as pd
import jcamp
import numpy as np
from PIL import Image
import io

# --- 1. Set up project paths ---
print("--- Setting up project paths for module imports ---")
GDRIVE_REPO_PATH = "/content/drive/MyDrive/colab_repos/CSU-IR"
CSU_IR_MODULE_PATH = os.path.join(GDRIVE_REPO_PATH, 'CSU-IR')
if CSU_IR_MODULE_PATH not in sys.path: sys.path.insert(0, CSU_IR_MODULE_PATH)
print("✅ Project paths set.")

# --- 2. Import custom modules ---
print("\n--- Importing custom modules ---")
from model.IR_encoder import IRModel
from model.SMILES_encoder import SmilesModel
from data_process.ir_process import preprocess_absorbances_spectra_higer_500, preprocess_absorbances_spectra_lower_500
from test_and_infer.test_and_infer_functions import get_feature_from_smiles
from test_and_infer.test_and_infer_functions import get_topK_result
from test_and_infer.test_and_infer_functions import normalize_smiles
print("✅ Custom modules imported.")

# --- 3. Define all file paths ---
print("\n--- Defining all file paths ---")
TOKENIZER_PATH = os.path.join(CSU_IR_MODULE_PATH, "model", "tokenizer-smiles-roberta-1e_new")

DEST_MODEL_I_WEIGHTS_PATH = os.path.join(CSU_IR_MODULE_PATH, 'check_points','Multi-stage_training_Stage_I_MD')
DEST_MODEL_II_WEIGHTS_PATH = os.path.join(CSU_IR_MODULE_PATH, 'check_points','Multi-stage_training_Stage_II_DFT')
DEST_MODEL_III_WEIGHTS_PATH = os.path.join(CSU_IR_MODULE_PATH, 'check_points','Multi-stage_training_Stage_III_EXP')

SMILES_MODEL_PATH_I = os.path.join(DEST_MODEL_I_WEIGHTS_PATH, "best_smiles_model_epoch_79_ratio_0.9810.pth")
IR_MODEL_PATH_I = os.path.join(DEST_MODEL_I_WEIGHTS_PATH, "best_ir_model_epoch_79_ratio_0.9810.pth")
SMILES_MODEL_PATH_II = os.path.join(DEST_MODEL_II_WEIGHTS_PATH, "best_smiles_model_epoch_71_ratio_0.9922.pth")
IR_MODEL_PATH_II = os.path.join(DEST_MODEL_II_WEIGHTS_PATH, "best_ir_model_epoch_71_ratio_0.9922.pth")
SMILES_MODEL_PATH_III = os.path.join(DEST_MODEL_III_WEIGHTS_PATH, "best_smiles_model_0.9230379746835443.pth")
IR_MODEL_PATH_III = os.path.join(DEST_MODEL_III_WEIGHTS_PATH, "best_ir_model_0.9230379746835443.pth")


TEST_DATA_PATH = os.path.join(CSU_IR_MODULE_PATH, "data", "test_data")

DEST_LIB_PS_PATH = os.path.join(CSU_IR_MODULE_PATH, "data", "processed_library", "PS")
SINGLE_TEST_DATA_PATH = os.path.join(CSU_IR_MODULE_PATH, "data", "example_library_and_ir_for_user_dinfined", "4-fluoro-ABUTINACA.JDX")
print("✅ File paths defined.")

# --- 4. Initialize all models and load weights ---
print("\n--- Initializing and loading all models ---")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Retrieval Models
ir_retrieval_model_I = IRModel()
smiles_retrieval_model_I = SmilesModel(roberta_model_path=None, roberta_tokenizer_path=TOKENIZER_PATH, feature_dim=768)
ir_retrieval_model_II = IRModel()
smiles_retrieval_model_II = SmilesModel(roberta_model_path=None, roberta_tokenizer_path=TOKENIZER_PATH, feature_dim=768)
ir_retrieval_model_III = IRModel()
smiles_retrieval_model_III = SmilesModel(roberta_model_path=None, roberta_tokenizer_path=TOKENIZER_PATH, feature_dim=768)

ir_retrieval_model_I.load_state_dict(torch.load(IR_MODEL_PATH_I, map_location=device))
smiles_retrieval_model_I.load_state_dict(torch.load(SMILES_MODEL_PATH_I, map_location=device))
ir_retrieval_model_II.load_state_dict(torch.load(IR_MODEL_PATH_II, map_location=device))
smiles_retrieval_model_II.load_state_dict(torch.load(SMILES_MODEL_PATH_II, map_location=device))
ir_retrieval_model_III.load_state_dict(torch.load(IR_MODEL_PATH_III, map_location=device))
smiles_retrieval_model_III.load_state_dict(torch.load(SMILES_MODEL_PATH_III, map_location=device))


ir_retrieval_model_I.to(device).eval()
smiles_retrieval_model_I.to(device).eval()
ir_retrieval_model_II.to(device).eval()
smiles_retrieval_model_II.to(device).eval()
ir_retrieval_model_III.to(device).eval()
smiles_retrieval_model_III.to(device).eval()
print("✅ Retrieval models loaded.")




# --- 5. Define helper functions ---
def load_gz_features(path):
    with gzip.open(path, 'rb') as f: return torch.load(f).to(torch.float32)
def load_smiles_list(path):
    with open(path, 'r', encoding='utf-8') as f: return [line.strip() for line in f if line.strip()]

# --- 6. Load all feature libraries and SMILES lists into memory ---
print("\n--- Loading all feature libraries (this may take a moment) ---")
# PS Libraries
library_existed_ps_features = load_gz_features(os.path.join(DEST_LIB_PS_PATH, 'library_Existing_PS_smiles_features_fp16.gz'))
library_derivative_ps_features = load_gz_features(os.path.join(DEST_LIB_PS_PATH, 'library_Derivative_PS_smiles_features_fp16.gz'))

ps_smiles_existed = load_smiles_list(os.path.join(DEST_LIB_PS_PATH, 'smiles_Existing_PS.txt'))
ps_smiles_derivative = load_smiles_list(os.path.join(DEST_LIB_PS_PATH, 'smiles_Derivative_PS.txt'))
print("✅ All libraries loaded.")

print("\n\n🎉🎉🎉 Global initialization complete! System is ready! 🎉🎉🎉")

### **Step 3: Performance**

In [ ]:
#@title 💊 Task 1: Batch Novel Psychoactive Substances (NPS) Retrieval
#@markdown This task searches for NPS samples against specialized libraries of known and derivative psychoactive substances.
#@markdown ---
#@markdown ### Select a library to search against:
library_to_search_ps = "Existing PS Library" #@param ["Existing PS Library", "Derivative PS Library"]

# --- Map user selection to actual data ---
library_map_ps = {
    "Existing PS Library": (library_existed_ps_features, ps_smiles_existed),
    "Derivative PS Library": (library_derivative_ps_features, ps_smiles_derivative),
}
selected_features_ps, selected_smiles_ps = library_map_ps[library_to_search_ps]

class IRSmilesDataset(Dataset):
    def __init__(self, ir_spectra, smiles):
        self.ir_spectra = ir_spectra
        self.smiles = smiles

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        return self.ir_spectra[idx], self.smiles[idx]

def batch_evaluate_retrieval_PS(loader, features_db, smiles_db, name):
    top_1, top_5, top_10, total = 0, 0, 0, 0
    with torch.no_grad():
        for ir_batch, smiles_batch in tqdm(loader, desc=f"Evaluating {name}"):
            ir_features = ir_retrieval_model_III(ir_batch.to(device))
            for i, ir_feature in enumerate(ir_features):
                original_smiles = normalize_smiles(smiles_batch[i])
                indices, _ = get_topK_result(ir_feature.unsqueeze(0), features_db, 10)
                for rank, lib_idx in enumerate(indices[0]):
                    if lib_idx < len(smiles_db) and original_smiles == normalize_smiles(smiles_db[lib_idx]):
                        if rank == 0: top_1 += 1
                        if rank < 5: top_5 += 1
                        if rank < 10: top_10 += 1
                        break
                total += 1
    print(f"\n--- Results for {name} ---\nRecall@1: {top_1/total:.4f} ({top_1}/{total})| Recall@5: {top_5/total:.4f} | Recall@10: {top_10/total:.4f}")

# --- Load NPS test data and run ---
if library_to_search_ps == "Derivative PS Library":
    NPS_test_smiles, NPS_test_ir = load_smiles_list(os.path.join(TEST_DATA_PATH, "NPS", "filtered_final_NPS_smiles.txt")), torch.load(os.path.join(TEST_DATA_PATH, "NPS","filtered_final_NPS_ir.pt"))
    NPS_loader = DataLoader(IRSmilesDataset(NPS_test_ir, NPS_test_smiles), batch_size=208, shuffle=False)
    batch_evaluate_retrieval_PS(NPS_loader, selected_features_ps, selected_smiles_ps, f"NPS Test in {library_to_search_ps}")
else:
    NPS_test_smiles, NPS_test_ir = load_smiles_list(os.path.join(TEST_DATA_PATH, "NPS", "NPS_smiles.txt")), torch.load(os.path.join(TEST_DATA_PATH, "NPS", "NPS_ir.pt"))
    NPS_loader = DataLoader(IRSmilesDataset(NPS_test_ir, NPS_test_smiles), batch_size=208, shuffle=False)
    batch_evaluate_retrieval_PS(NPS_loader, selected_features_ps, selected_smiles_ps, f"NPS Test in {library_to_search_ps}")

In [ ]:
#@title 💊 Task 2: Single New Psychoactive Substances (NPS) Retrieval
#@markdown This task searches for single NPS samples against specialized libraries of known and derivative psychoactive substances, using 4-fluoro-ABUTINACA as an example.
#@markdown ---
#@markdown ### Select a library to search against:
library_to_search_ps = "Derivative PS Library" #@param ["Existing PS Library", "Derivative PS Library"]
target_smiles = 'O=C(NC12CC3CC(CC(C3)C1)C2)c1nn(CCCCF)c2ccccc12'
# --- Map user selection to actual data ---
library_map_ps = {
    "Existing PS Library": (library_existed_ps_features, ps_smiles_existed),
    "Derivative PS Library": (library_derivative_ps_features, ps_smiles_derivative),
}
selected_features_ps, selected_smiles_ps = library_map_ps[library_to_search_ps]

def draw_molecules(target_smiles, smiles_list, scores_list, mols_per_row=5):
    if len(smiles_list) != len(scores_list):
        raise ValueError("The lengths of smiles_list and scores_list must be the same!")
    target_mol = Chem.MolFromSmiles(target_smiles)
    if not target_mol:
        raise ValueError("The provided target_smiles is invalid!")
    target_legend = "Target Molecule"
    candidate_mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    candidate_legends = [f"Top {i + 1}: {round(score, 4)}" for i, score in enumerate(scores_list)]
    valid_mols_legends = [
        (mol, legend) for mol, legend in zip(candidate_mols, candidate_legends) if mol is not None
    ]
    if valid_mols_legends:
        valid_candidate_mols, valid_candidate_legends = zip(*valid_mols_legends)
    else:
        valid_candidate_mols, valid_candidate_legends = [], []
    all_mols_to_draw = [target_mol] + [None] * (mols_per_row - 1) + list(valid_candidate_mols)
    all_legends_to_draw = [target_legend] + [""] * (mols_per_row - 1) + list(valid_candidate_legends)

    img = Draw.MolsToGridImage(
        all_mols_to_draw,
        molsPerRow=mols_per_row,
        subImgSize=(300, 300),
        legends=all_legends_to_draw
    )

    return img

def single_retrieval_library(ir_feature, library_features, library_smiles):
    indices, scores = get_topK_result(ir_feature, library_features, 10)

    top_smiles = []
    top_scores = []

    for sco, idx in zip(scores[0], indices[0]):
        retrieved_smiles = library_smiles[idx.item()]
        top_smiles.append(retrieved_smiles)
        top_scores.append(sco.item())

    img = draw_molecules(target_smiles, top_smiles,top_scores)
    return img, top_scores, top_smiles

def single_retrieval(ir_spectra_file, spectrum_type, library_features, library_smiles):
    if hasattr(ir_spectra_file, 'name'):
        file_path = ir_spectra_file.name
    else:
        file_path = ir_spectra_file
    print(f"Processing file: {file_path}")
    if file_path.lower().endswith('.csv'):
        df = pd.read_csv(file_path)
        wavenumbers = df.iloc[:, 0].values
        transmittances = df.iloc[:, 1].values
    elif file_path.lower().endswith('.jdx'):
        data = jcamp.jcamp_readfile(file_path)
        wavenumbers = np.array(data['x'], dtype=float)
        transmittances = np.array(data['y'], dtype=float)
    else:
        raise ValueError("Unsupported file format. Please upload a CSV or JDX file.")

    if spectrum_type == "absorbance spectrum":
        if wavenumbers[0] > 500:
            ir_data = preprocess_absorbances_spectra_higer_500(wavenumbers, transmittances)
        else:
            ir_data = preprocess_absorbances_spectra_lower_500(file_path, wavenumbers, transmittances)
    else:
        transmittances = transmittances / 100.0
        if wavenumbers[0] > 500:
            ir_data = preprocess_absorbances_spectra_higer_500(wavenumbers, transmittances)
        else:
            ir_data = preprocess_absorbances_spectra_lower_500(file_path, wavenumbers, transmittances)

    ir_spectra_tensor = torch.tensor(ir_data, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        ir_feature = ir_retrieval_model_III(ir_spectra_tensor)

    img, scores,top_smiles = single_retrieval_library(ir_feature, library_features, library_smiles)
    return img, scores,top_smiles

ir_spectra_file = SINGLE_TEST_DATA_PATH
img, scores, top_smiles = single_retrieval(ir_spectra_file , spectrum_type="absorbance spectrum", library_features = selected_features_ps, library_smiles = selected_smiles_ps)

# --- Display the results ---
if img:
    print("\n--- Retrieval Results ---")
    for i, (smile, score) in enumerate(zip(top_smiles, scores)):
        print(f"Top {i+1}: Score=  {score:.4f}, SMILES=  {smile}")

    print("\n--- Top 10 Candidate Molecules ---")
    # By placing the image object as the last line of the cell, Colab will automatically display it.
    display(img)
else:
    print("\nNo image to display. Please run the cell again and provide a file.")

In [ ]:
#@title 💊 Task 3: Test result for simulated data
#@markdown ---
#@markdown ### Select one simulated library to reproduce test set results :
library_to_search_ps = "QM9-like molecules in QMe14S(excluding QM9S)" #@param ["QM9S", "QM9-like molecules in QMe14S(excluding QM9S)"]



class IRSmilesDataset(Dataset):
    def __init__(self, ir_spectra, smiles):
        self.ir_spectra = ir_spectra
        self.smiles = smiles

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        return self.ir_spectra[idx], self.smiles[idx]

def smiles_encode(smilesmodel,smiles_str):
    with torch.no_grad():
       tokenizer = smilesmodel.smiles_tokenizer
       encoded_smiles = [tokenizer.encode_plus(
                text=smiles,
                max_length=300,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            ) for smiles in smiles_str]

       input_ids = torch.cat([item['input_ids'] for item in encoded_smiles], dim=0).to(device)
       attention_mask = torch.cat([item['attention_mask'] for item in encoded_smiles], dim=0).to(device)
       lengths = attention_mask.sum(dim=1)

       smiles_feature = smilesmodel.encode((input_ids, attention_mask),lengths)
       return smiles_feature

def get_feature_from_smiles(smiles_list, smilesmodel, batch_size=288):
    contexts = []
    print("start load batch")
    for i in range(0, len(smiles_list), batch_size):
        contexts.append(smiles_list[i:i + batch_size])
    print("start encode batch")
    result = [smiles_encode(smilesmodel,i).cpu() for i in tqdm(contexts)]
    result = torch.cat(result, 0)
    return result

def batch_evaluate_retrieval(loader, features_db, smiles_db, ir_retrieval_model, name):
    top_1, top_5, top_10, total = 0, 0, 0, 0
    with torch.no_grad():
        for ir_batch, smiles_batch in tqdm(loader, desc=f"Evaluating {name}"):
            ir_features = ir_retrieval_model(ir_batch.to(device))
            for i, ir_feature in enumerate(ir_features):
                original_smiles = normalize_smiles(smiles_batch[i])
                indices, _ = get_topK_result(ir_feature.unsqueeze(0), features_db, 10)
                for rank, lib_idx in enumerate(indices[0]):
                    if lib_idx < len(smiles_db) and original_smiles == normalize_smiles(smiles_db[lib_idx]):
                        if rank == 0: top_1 += 1
                        if rank < 5: top_5 += 1
                        if rank < 10: top_10 += 1
                        break
                total += 1
    print(f"\n--- Results for {name} ---\nRecall@1: {top_1/total:.4f} ({top_1}/{total})| Recall@5: {top_5/total:.4f} | Recall@10: {top_10/total:.4f}")



# --- Load NPS test data and run ---
#if library_to_search_ps == "Multimodal_Spectroscopic_Library":
    #Multimodal_Spectroscopic_Library_test_smiles, Multimodal_Spectroscopic_Library_test_ir = load_smiles_list(os.path.join(TEST_DATA_PATH, "Multimodal_Spectroscopic_Library", "Multimodal_Spectroscopic_Library_test_smiles.txt")), torch.load(os.path.join(TEST_DATA_PATH, "Multimodal_Spectroscopic_Library", "Multimodal_Spectroscopic_Library_ir.pt"))
    #Multimodal_Spectroscopic_Library_loader = DataLoader(IRSmilesDataset(Multimodal_Spectroscopic_Library_test_ir, Multimodal_Spectroscopic_Library_test_smiles), batch_size=208, shuffle=False)
    #features_db = get_feature_from_smiles(Multimodal_Spectroscopic_Library_test_smiles, smiles_retrieval_model_I, batch_size=288)
    #smiles_db = Multimodal_Spectroscopic_Library_test_smiles
    #batch_evaluate_retrieval(Multimodal_Spectroscopic_Library_loader, features_db, smiles_db, ir_retrieval_model_I, f"Multimodal_Spectroscopic_Library Test Result")
if library_to_search_ps == "QM9S":
    QM9S_test_smiles, QM9S_test_ir = load_smiles_list(os.path.join(TEST_DATA_PATH, "QM9S", "QM9S_DFT_test_smiles.txt")), torch.load(os.path.join(TEST_DATA_PATH, "QM9S", "QM9S_DFT_test_ir.pt"))
    QM9S_loader = DataLoader(IRSmilesDataset(QM9S_test_ir, QM9S_test_smiles), batch_size=208, shuffle=False)
    features_db = get_feature_from_smiles(QM9S_test_smiles, smiles_retrieval_model_II, batch_size=288)
    smiles_db = QM9S_test_smiles
    batch_evaluate_retrieval(QM9S_loader, features_db, smiles_db, ir_retrieval_model_II, f"QM9S Test Result")
elif library_to_search_ps == "QM9-like molecules in QMe14S(excluding QM9S)":
    QMe14S_test_smiles, QMe14S_test_ir = load_smiles_list(os.path.join(TEST_DATA_PATH, "QMe14S", "QM9-like molecules in QMe14S(excluding QM9S).txt")), torch.load(os.path.join(TEST_DATA_PATH, "QMe14S", "QM9-like molecules in QMe14S(excluding QM9S).pt"))
    QMe14S_loader = DataLoader(IRSmilesDataset(QMe14S_test_ir, QMe14S_test_smiles), batch_size=208, shuffle=False)
    features_db = get_feature_from_smiles(QMe14S_test_smiles, smiles_retrieval_model_II, batch_size=288)
    smiles_db = QMe14S_test_smiles
    batch_evaluate_retrieval(QMe14S_loader, features_db, smiles_db, ir_retrieval_model_II, f"QM9-like molecules in QMe14S(excluding QM9S) Test Result")